# Day 4 — Clean Continuous Pipeline (No Bronze)

**Goal:** Predict flood risk using a clean Medallion architecture where raw data is purged after processing.

**Architecture:** 
1. **Ingestion**: Live forecast is processed in-memory.
2. **Persistence**: Only standardized Silver and engineered Gold layers are saved to disk.
3. **Context**: 6-hour resampled stream ensures continuity without storing raw bytes.

In [1]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path
sys.path.append(os.path.abspath('../'))
from src import config, ingestion, pipeline

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

## 1. Fetch Live Forecast Data

In [2]:
# Fetch 15-day forecast for all Baku zones
forecast_frames = []
for zone in config.BAKU_ZONES:
    df_z = ingestion.fetch_weather_forecast(zone['zone'], zone['latitude'], zone['longitude'], forecast_days=15)
    forecast_frames.append(df_z)

df_forecast_raw = pd.concat(forecast_frames)
print(f"Live forecast data fetched: {len(df_forecast_raw)} hourly rows.")

2026-04-25 16:54:59,745 - INFO - Fetching 15-day forecast for High Relief
2026-04-25 16:55:00,331 - INFO - Fetching 15-day forecast for Low Relief
2026-04-25 16:55:00,963 - INFO - Fetching 15-day forecast for Moderate Relief


Live forecast data fetched: 1080 hourly rows.


## 2. Execute Clean Pipeline
This step resamples data to Silver and purges raw Bronze data from disk.

In [3]:
# Process live data through the clean pipeline
# Note: Bronze tables are not persisted in this mode.
df_features = pipeline.run_forecast_pipeline(df_forecast_raw)

print(f"Pipeline complete. {len(df_features)} rows generated for prediction window.")
df_features.head()

Pipeline complete. 171 rows generated for prediction window.


,zone,time_6h,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_temperature_0_to_7cm,et0_fao_evapotranspiration,river_discharge,precip_lag_6h,precip_lag_12h,precip_lag_24h,precip_lag_48h,discharge_lag_6h,discharge_lag_24h,temp_lag_24h,precip_roll_sum_24h,precip_roll_sum_48h,precip_roll_sum_72h,precip_roll_max_24h,discharge_roll_max_24h,humidity_roll_max_24h,et0_roll_sum_24h,api_7d,soil_saturation_index,soil_moisture_change_6h,soil_moisture_deficit,frozen_ground_flag,discharge_trend_6h,temp_trend_24h,et_deficit_6h,et_deficit_24h,humidity_precip_product,highland_precip_24h,highland_discharge_6h,zone_cascade_risk,hour_sin,hour_cos,doy_sin,doy_cos,is_winter,is_flood
0,High Relief,2026-04-25 18:00:00,9.716667,85.333333,0.0,30.900000,0.155000,0.282833,13.383333,0.24,0.0,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,86.666667,2.56,0.0,0.218917,-0.006333,0.231083,0,0.0,NaN,-0.24,-2.56,0.0,0.0,0.0,0.0,-1.000000e+00,-1.836970e-16,0.917584,-0.397543,0,0
1,High Relief,2026-04-26 00:00:00,7.850000,90.166667,0.0,5.133333,0.158833,0.281667,9.816667,0.00,0.0,0.0,0.0,0.0,NaN,0.0,0.0,9.800000,0.0,0.0,0.0,0.0,0.0,90.166667,2.49,0.0,0.220250,0.003833,0.229750,0,0.0,-1.950000,0.00,-2.49,0.0,0.0,0.0,0.0,0.000000e+00,1.000000e+00,0.910605,-0.413279,0,0
2,High Relief,2026-04-26 06:00:00,10.633333,78.000000,0.0,8.733333,0.160667,0.281000,11.416667,0.88,0.0,0.0,0.0,0.0,NaN,0.0,0.0,11.700000,0.0,0.0,0.0,0.0,0.0,90.166667,2.71,0.0,0.220833,0.001833,0.229167,0,0.0,-1.066667,-0.88,-2.71,0.0,0.0,0.0,0.0,1.000000e+00,6.123234e-17,0.910605,-0.413279,0,0
3,High Relief,2026-04-26 12:00:00,16.350000,66.166667,0.0,23.800000,0.148833,0.279833,16.883333,2.14,0.0,0.0,0.0,0.0,NaN,0.0,0.0,14.183333,0.0,0.0,0.0,0.0,0.0,90.166667,3.26,0.0,0.214333,-0.011833,0.235667,0,0.0,2.166667,-2.14,-3.26,0.0,0.0,0.0,0.0,1.224647e-16,-1.000000e+00,0.910605,-0.413279,0,0
4,High Relief,2026-04-26 18:00:00,13.733333,81.666667,0.0,18.950000,0.144500,0.278833,13.466667,0.31,0.0,0.0,0.0,0.0,NaN,0.0,0.0,9.716667,0.0,0.0,0.0,0.0,0.0,90.166667,3.33,0.0,0.211667,-0.004333,0.238333,0,0.0,4.016667,-0.31,-3.33,0.0,0.0,0.0,0.0,-1.000000e+00,-1.836970e-16,0.910605,-0.413279,0,0


## 3. Predict & Alert

In [4]:
checkpoint = joblib.load('../models/baku_sentinel_rf.joblib')
model = checkpoint['model']
feature_cols = checkpoint['features']

# Prepare and Sync features
X_live = pd.get_dummies(df_features.drop(columns=['time_6h', 'river_discharge', 'is_flood']), columns=['zone'], drop_first=True)
for col in feature_cols: 
    if col not in X_live.columns: X_live[col] = 0
X_live = X_live[feature_cols].fillna(0)

# Calculate Inundation Probability
df_features['risk_score'] = model.predict_proba(X_live)[:, 1]

# Summary Alert
display(df_features.groupby('zone').agg({'risk_score': ['max', 'mean']}))

risk_score          
                       max      mean
zone                                
High Relief           0.00  0.000000
Low Relief            0.01  0.000175
Moderate Relief       0.00  0.000000